# 02. 탐색 · 직무 매칭 확정 → `data/curated/jobs.csv`

**목적:** 시연 직무를 정하고, 직무마다 공공 영상 ID · NCS 세분류 코드 · 장애인 직종코드를 확정한다.

결정 기준
- 공공 직업정보 영상이 있는 직무 (카페·바리스타는 없음). 후보: 편의점정리원 15123185, 주방보조원 15123777
- NCS 세분류와 직종코드가 각각 1개로 좁혀지는가
- 결정 근거는 `note` 열과 `docs/data-report.md` 에 남긴다

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent            # notebooks/ 에서 실행한다고 가정
sys.path.insert(0, str(ROOT / "backend"))
DATA = ROOT / "data"
import pandas as pd
pd.set_option("display.max_colwidth", 80)

jobcode = pd.read_csv(DATA / "raw/api/jobcode.csv", dtype=str)
ncs = pd.read_csv(DATA / "raw/api/ncs_classification.csv", dtype=str)

In [ ]:
def find(df, kw, cols):
    m = False
    for c in cols:
        m = m | df[c].fillna("").str.contains(kw)
    return df[m]

# 직종 후보 (열 이름은 01 에서 확인한 실제 이름으로 바꾼다)
find(jobcode, "주방|조리|편의점|진열|판매", ["jobCdNm", "simlrJob"])

In [ ]:
# NCS 세분류 후보 (열 이름은 실제 응답 기준으로 바꾼다)
find(ncs, "조리|판매|매장|음료", [c for c in ncs.columns if c.endswith("Nm")])

## 고른 NCS 세분류의 능력단위·수행준거 받기

세분류 코드(dutyCd)를 정한 뒤 실행한다. 매뉴얼 단계와 대조할 기준이 된다 (인계서 4.2, 5.4).

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
from app.ingest.data_go_kr import DataGoKrClient, fetch_ncs_comp_units, fetch_ncs_ksa
client = DataGoKrClient(os.environ["DATA_GO_KR_KEY"])

DUTY_CD = ""   # 위 검색으로 고른 세분류 코드
units = pd.DataFrame(list(fetch_ncs_comp_units(client, DUTY_CD)))
ksa = pd.DataFrame(list(fetch_ncs_ksa(client, DUTY_CD)))
units.to_csv(DATA / f"raw/api/ncs_units_{DUTY_CD}.csv", index=False, encoding="utf-8-sig")
ksa.to_csv(DATA / f"raw/api/ncs_ksa_{DUTY_CD}.csv", index=False, encoding="utf-8-sig")
print(units.shape, ksa.shape)
ksa.head(20)   # gbnName 에 어떤 값(수행준거·지식·기술·태도)이 오는지 data-report.md 에 기록 (12장 #7)

## 확정 결과 기록

In [ ]:
jobs = pd.DataFrame([
    # job_key, job_name, public_video_id, video_file, ncs_duty_cd, job_cd, status, note
    ["kitchen_helper", "주방보조원", "15123777", "kitchen_helper.mp4", "", "", "candidate", ""],
], columns=["job_key", "job_name", "public_video_id", "video_file", "ncs_duty_cd", "job_cd", "status", "note"])
jobs.to_csv(DATA / "curated/jobs.csv", index=False, encoding="utf-8-sig")
jobs

In [ ]:
from app.data_check import check_all
check_all(DATA) or "통과"